# VeReMi Graph Construction Engine — All 225 Archives

This notebook processes the original VeReMi `.tgz` archives into provenance-preserving graph tables.

**Core representation**

- Node: vehicle.
- Node state: type-2 local state.
- Directed edge: a type-3 received V2X message, `sender → receiver`.
- Edge target: `attackerType` matched from type-4 ground truth.
- Edge features: send/receive times, communication delay, RSSI, reported motion state.
- Graphs: temporal snapshots within each simulation archive.

The notebook never creates edges across different `.tgz` archives.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ROOT, OUTPUT_ROOT, WINDOW_SIZE,
    TIME_TOLERANCE, SAVE_PER_ARCHIVE,
    PARQUET_COMPRESSION
)
from src.veremi_graph_engine import (
    parse_archive,
    match_ground_truth,
    build_edge_table,
    build_node_state_table,
    assign_snapshot_ids,
    archive_summary,
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "archives").mkdir(parents=True, exist_ok=True)

tgz_files = sorted(ROOT.glob("*.tgz"))

print("Archives found:", len(tgz_files))
print("Window size:", WINDOW_SIZE, "seconds")
print("Output:", OUTPUT_ROOT)

if not tgz_files:
    raise FileNotFoundError(f"No .tgz files found in {ROOT}")


In [ ]:
# ============================================================
# 1. DRY RUN ON FIRST ARCHIVE
# ============================================================

archive = tgz_files[0]

logs_meta, local_states, received, gt = parse_archive(archive)
matched, duplicate_keys = match_ground_truth(
    received, gt, TIME_TOLERANCE
)
edges = build_edge_table(matched)
nodes = build_node_state_table(local_states)
edges = assign_snapshot_ids(edges, WINDOW_SIZE)

summary = archive_summary(
    archive, logs_meta, local_states,
    received, gt, matched, edges,
    duplicate_keys
)

display(pd.DataFrame([summary]))
display(edges.head())
display(nodes.head())

print("Snapshots:", edges["snapshot_id"].nunique())


In [ ]:
# ============================================================
# 2. PROCESS ALL ARCHIVES
# ============================================================

all_summaries = []
failed_archives = []

for idx, archive in enumerate(tgz_files, start=1):

    print(f"[{idx}/{len(tgz_files)}] {archive.name}")

    try:
        logs_meta, local_states, received, gt = parse_archive(archive)

        matched, duplicate_keys = match_ground_truth(
            received, gt, TIME_TOLERANCE
        )

        edges = build_edge_table(matched)
        nodes = build_node_state_table(local_states)
        edges = assign_snapshot_ids(edges, WINDOW_SIZE)

        summary = archive_summary(
            archive, logs_meta, local_states,
            received, gt, matched, edges,
            duplicate_keys
        )

        summary["snapshot_count"] = (
            int(edges["snapshot_id"].nunique())
            if len(edges) else 0
        )

        all_summaries.append(summary)

        if SAVE_PER_ARCHIVE:
            archive_dir = (
                OUTPUT_ROOT / "archives" / archive.stem
            )
            archive_dir.mkdir(parents=True, exist_ok=True)

            logs_meta.to_parquet(
                archive_dir / "log_metadata.parquet",
                index=False,
                compression=PARQUET_COMPRESSION
            )

            nodes.to_parquet(
                archive_dir / "node_states.parquet",
                index=False,
                compression=PARQUET_COMPRESSION
            )

            edges.to_parquet(
                archive_dir / "communication_edges.parquet",
                index=False,
                compression=PARQUET_COMPRESSION
            )

            # Compact snapshot index
            snapshot_summary = (
                edges.groupby("snapshot_id")
                .agg(
                    snapshot_start=("snapshot_start", "first"),
                    snapshot_end=("snapshot_end", "first"),
                    edge_count=("sender", "size"),
                    unique_senders=("sender", "nunique"),
                    unique_receivers=("receiver", "nunique"),
                    attack_classes=("attackerType", "nunique")
                )
                .reset_index()
            )

            snapshot_summary.to_parquet(
                archive_dir / "snapshot_summary.parquet",
                index=False,
                compression=PARQUET_COMPRESSION
            )

    except Exception as e:
        failed_archives.append({
            "archive_name": archive.name,
            "error": repr(e)
        })
        print("FAILED:", e)

summary_df = pd.DataFrame(all_summaries)
failed_df = pd.DataFrame(failed_archives)

summary_df.to_csv(
    OUTPUT_ROOT / "all_archive_summary.csv",
    index=False
)

failed_df.to_csv(
    OUTPUT_ROOT / "failed_archives.csv",
    index=False
)

print("\nCompleted archives:", len(summary_df))
print("Failed archives:", len(failed_df))
display(summary_df.head())
display(failed_df.head())


In [ ]:
# ============================================================
# 3. GLOBAL VALIDATION
# ============================================================

if summary_df.empty:
    raise RuntimeError("No archives were processed successfully.")

validation = pd.DataFrame({
    "metric": [
        "archives_processed",
        "archives_failed",
        "total_log_files",
        "total_local_state_rows",
        "total_received_messages",
        "total_ground_truth_rows",
        "total_communication_edges",
        "total_snapshots",
        "minimum_gt_match_percent",
        "median_gt_match_percent",
        "minimum_gt_time_valid_percent",
        "total_duplicate_gt_keys"
    ],
    "value": [
        len(summary_df),
        len(failed_df),
        summary_df["log_files"].sum(),
        summary_df["local_state_rows"].sum(),
        summary_df["received_message_rows"].sum(),
        summary_df["ground_truth_rows"].sum(),
        summary_df["communication_edges"].sum(),
        summary_df["snapshot_count"].sum(),
        summary_df["gt_match_percent"].min(),
        summary_df["gt_match_percent"].median(),
        summary_df["gt_time_valid_percent"].min(),
        summary_df["gt_duplicate_sender_message_keys"].sum()
    ]
})

display(validation)

validation.to_csv(
    OUTPUT_ROOT / "global_validation_summary.csv",
    index=False
)


In [ ]:
# ============================================================
# 4. ATTACK-TYPE DISTRIBUTION ACROSS ALL EDGE TABLES
# ============================================================

import duckdb

edge_glob = (
    OUTPUT_ROOT / "archives" / "*" /
    "communication_edges.parquet"
).as_posix()

con = duckdb.connect()

attack_distribution = con.execute(f"""
    SELECT
        attackerType,
        COUNT(*) AS edge_count,
        COUNT(DISTINCT archive_name) AS archives_present
    FROM read_parquet('{edge_glob}', union_by_name=true)
    GROUP BY attackerType
    ORDER BY edge_count DESC
""").df()

display(attack_distribution)

attack_distribution.to_csv(
    OUTPUT_ROOT / "edge_attack_type_distribution.csv",
    index=False
)


In [ ]:
# ============================================================
# 5. GRAPH SIZE DISTRIBUTION
# ============================================================

graph_size_summary = summary_df[
    [
        "archive_name",
        "unique_vehicles",
        "communication_edges",
        "snapshot_count",
        "median_comm_delay",
        "gt_match_percent"
    ]
].copy()

display(graph_size_summary.describe(include="all"))

graph_size_summary.to_csv(
    OUTPUT_ROOT / "graph_size_summary.csv",
    index=False
)


In [ ]:
# ============================================================
# 6. IMPORTANT RESEARCH CHECK:
# IS THE TARGET EDGE-LEVEL?
# ============================================================

print(
    "attackerType originates from matched type-4 transmitted-message "
    "ground truth. Therefore the most natural first GNN task is "
    "MESSAGE/EDGE CLASSIFICATION, not node classification."
)

print(
    "\nRecommended next phase after validation:\n"
    "1. Create temporal snapshots.\n"
    "2. Attach latest type-2 state to each vehicle node.\n"
    "3. Use type-3 communications as directed edges.\n"
    "4. Predict attackerType on each message edge.\n"
    "5. Benchmark MLP/Edge-MLP vs GCN/GraphSAGE/GAT edge classifiers."
)
